## 1. Общий setup

Заполните `REPO_URL`, при необходимости поменяйте `BRANCH`, затем выполните блок. CSV-файлы должны лежать в корне репозитория.

In [ ]:
# 1. Общий setup
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Timur713/HSE_RAG_Hackaton"  # pure URL or markdown link are both accepted
BRANCH = "main"
PROJECT_DIR = Path("/content/legal_hse_4")
RUN_OPTIONAL_DENSE = False  # True installs sentence-transformers and enables dense configs


def normalize_repo_url(value: str) -> str:
    value = str(value).strip()
    markdown_match = re.search(r"\((https?://[^)]+|git@[^)]+)\)", value)
    if markdown_match:
        value = markdown_match.group(1)
    else:
        plain_match = re.search(r"https?://[^\s\]]+|git@[^\s\]]+", value)
        if plain_match:
            value = plain_match.group(0)
    value = value.strip().strip("'").strip('"')
    if value.endswith("/"):
        value = value[:-1]
    return value


def run_cmd(cmd, *, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


REPO_URL = normalize_repo_url(REPO_URL)
if not REPO_URL:
    raise ValueError("Заполните REPO_URL чистым URL GitHub-репозитория.")
print("Using repo:", REPO_URL)

# Colab workspace is disposable. Re-cloning is more reliable than trying to repair a dirty git checkout.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

clone_result = run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=False)
if clone_result.returncode != 0:
    print(f"Clone with --branch {BRANCH!r} failed; retrying default branch clone.")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    branches = run_cmd(["git", "-C", str(PROJECT_DIR), "branch", "-a"], check=False)
    if f"remotes/origin/{BRANCH}" in branches.stdout:
        run_cmd(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH])
    else:
        print(f"Branch {BRANCH!r} not found; using repository default branch.")

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
if RUN_OPTIONAL_DENSE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from legal_hse.config import PathConfig

paths = PathConfig.from_root(PROJECT_DIR)
paths.ensure_dirs()
paths.require_input_files()
print(f"Project ready: {PROJECT_DIR}")


## 2. Все эксперименты

`cv` используйте для перебора конфигураций на train-pool. `holdout` запускайте отдельно только для финальной проверки выбранных экспериментов.

In [ ]:
# 2a. Focused sweep: токенизация + лемматизация для sparse retrieval
import sys
from legal_hse.experiments import default_experiments

# pymorphy3 нужен только для lemmatize=True; без него код fallback'ится в обычные токены.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])

TOKENIZATION_EXPERIMENT_NAMES = [
    # controls from the current strong sparse baseline
    "bm25_doc",
    "bm25_chunk_line_10_5_max",
    "tfidf_char_doc_3_5",
    "rrf_sparse_doc_chunk_char",
    # pure lemmatization checks
    "tfidf_word_lemma_doc",
    "bm25_lemma_doc",
    # legal-aware tokenizer: keeps ст. 333.19, 75-ФЗ, НК/ГК/РФ and removes ФИО/адрес noise
    "tfidf_word_legal_lemma_doc",
    "bm25_legal_lemma_doc",
    "bm25_legal_lemma_chunk_line_10_5_max",
    "rrf_legal_lemma_doc_chunk",
    "rrf_sparse_legal_lemma_char",
    "rrf_sparse_legal_lemma_word_char",
    # phrase variant for stable legal collocations; keep only if validation beats simpler lemmas
    "bm25_legal_phrase_doc",
    "bm25_legal_phrase_chunk_line_10_5_max",
    "rrf_legal_phrase_doc_chunk",
]

available_experiments = {spec.name for spec in default_experiments(include_optional=RUN_OPTIONAL_DENSE)}
missing = sorted(set(TOKENIZATION_EXPERIMENT_NAMES) - available_experiments)
if missing:
    raise ValueError(f"Unknown experiments: {missing}")

MODE = "cv"
EXPERIMENT_NAMES = TOKENIZATION_EXPERIMENT_NAMES
SEED = 42
N_SPLITS = 5
print("Tokenization/lemmatization sweep:")
for name in EXPERIMENT_NAMES:
    print("-", name)


In [ ]:
# 2b. Recall@20/30/50/100 diagnostics + candidate-generation sweep (single-cell run)
# This cell runs the full pre-rerank recall workflow:
# 1) oracle diagnostics for current branches;
# 2) deeper chunk retrieval;
# 3) chunk-view sweep;
# 4) BM25 k1/b sweep;
# 5) quota/union fusion;
# 6) optional dense candidate generation.

import sys

from legal_hse.experiments import (
    recall_candidate_experiments,
    run_recall_oracle_diagnostics,
    run_suite,
    select_best_experiment,
)

# pymorphy3 is required for the lemmatized sparse configs to be reproducible in Colab.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])

# Dense is enabled by default for this recall sweep because point 5 is part of the plan.
# Set ENABLE_RECALL_DENSE = False before running this cell to run sparse-only.
# Set ENABLE_BGE_M3 = True before running this cell to add BGE-M3 experiments too.
ENABLE_RECALL_DENSE = bool(globals().get("ENABLE_RECALL_DENSE", True))
ENABLE_BGE_M3 = bool(globals().get("ENABLE_BGE_M3", False))
if ENABLE_RECALL_DENSE and not RUN_OPTIONAL_DENSE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])
    RUN_OPTIONAL_DENSE = True

RECALL_MODE = globals().get("RECALL_MODE", "cv")  # "cv", "holdout", or "train"
RECALL_SEED = globals().get("RECALL_SEED", 42)
RECALL_N_SPLITS = globals().get("RECALL_N_SPLITS", 5)
RECALL_EVAL_KS = tuple(globals().get("RECALL_EVAL_KS", (5, 10, 20, 30, 50, 100)))
RECALL_EVAL_DEPTH = max(globals().get("RECALL_EVAL_DEPTH", 100), max(RECALL_EVAL_KS))
BASELINE_FUSION = "rrf_sparse_legal_lemma_char"

EXTRA_RECALL_EXPERIMENTS = recall_candidate_experiments(
    include_optional=ENABLE_RECALL_DENSE,
    include_bge_m3=ENABLE_BGE_M3,
)
RECALL_EXPERIMENT_NAMES = [BASELINE_FUSION] + [spec.name for spec in EXTRA_RECALL_EXPERIMENTS]

print("Step 1/2: oracle diagnostics for the current sparse candidate branches")
RECALL_DIAGNOSTICS = run_recall_oracle_diagnostics(
    data_dir=PROJECT_DIR,
    branch_names=[
        "bm25_legal_lemma_doc",
        "bm25_legal_lemma_chunk_line_10_5_max",
        "tfidf_char_doc_3_5",
    ],
    baseline_fusion_name=BASELINE_FUSION,
    mode=RECALL_MODE,
    include_optional=ENABLE_RECALL_DENSE,
    extra_experiments=EXTRA_RECALL_EXPERIMENTS,
    seed=RECALL_SEED,
    n_splits=RECALL_N_SPLITS,
    depth=RECALL_EVAL_DEPTH,
    eval_ks=RECALL_EVAL_KS,
)
display(RECALL_DIAGNOSTICS)

print(f"Step 2/2: running {len(RECALL_EXPERIMENT_NAMES)} recall experiments at k={RECALL_EVAL_KS}")
summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=RECALL_EXPERIMENT_NAMES,
    extra_experiments=EXTRA_RECALL_EXPERIMENTS,
    mode=RECALL_MODE,
    include_optional=ENABLE_RECALL_DENSE,
    seed=RECALL_SEED,
    n_splits=RECALL_N_SPLITS,
    eval_depth=RECALL_EVAL_DEPTH,
    eval_ks=RECALL_EVAL_KS,
    comparison_baseline=BASELINE_FUSION,
)

cols = [
    "experiment",
    "status",
    "priority",
    "n_splits",
    "valid_recall@5_mean",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@30_mean",
    "valid_recall@50_mean",
    "valid_recall@100_mean",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "holdout_recall@5_mean",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@30_mean",
    "holdout_recall@50_mean",
    "holdout_recall@100_mean",
    "duration_sec",
]
visible_cols = [c for c in cols if c in summary.columns]
sort_col = next(
    c
    for c in [
        "holdout_recall@50_mean",
        "valid_recall@50_mean",
        "holdout_recall@100_mean",
        "valid_recall@100_mean",
        "holdout_recall@30_mean",
        "valid_recall@30_mean",
        "holdout_recall@20_mean",
        "valid_recall@20_mean",
        "holdout_recall@5_mean",
        "valid_recall@5_mean",
    ]
    if c in summary.columns
)
ranked_summary = summary[visible_cols].sort_values(["status", sort_col], ascending=[True, False])
display(ranked_summary.head(40))

BEST_RECALL50_EXPERIMENT = select_best_experiment(summary, metric=sort_col)
BEST_RECALL30_EXPERIMENT = select_best_experiment(
    summary,
    metric=next(c for c in ["holdout_recall@30_mean", "valid_recall@30_mean", sort_col] if c in summary.columns),
)
BEST_RECALL20_EXPERIMENT = select_best_experiment(
    summary,
    metric=next(c for c in ["holdout_recall@20_mean", "valid_recall@20_mean", sort_col] if c in summary.columns),
)
BEST_RECALL100_EXPERIMENT = select_best_experiment(
    summary,
    metric=next(c for c in ["holdout_recall@100_mean", "valid_recall@100_mean", sort_col] if c in summary.columns),
)
BEST_EXPERIMENT = select_best_experiment(summary)  # Keeps the submission cell Recall@5-oriented by default.

print("BEST_RECALL100_EXPERIMENT =", BEST_RECALL100_EXPERIMENT)
print("BEST_RECALL50_EXPERIMENT =", BEST_RECALL50_EXPERIMENT)
print("BEST_RECALL30_EXPERIMENT =", BEST_RECALL30_EXPERIMENT)
print("BEST_RECALL20_EXPERIMENT =", BEST_RECALL20_EXPERIMENT)
print("BEST_EXPERIMENT [Recall@5 default] =", BEST_EXPERIMENT)
print("Full summary saved under:", PROJECT_DIR / "reports")

## 2c. Отдельный BGE-M3 retrieval experiment

Эта ячейка проверяет `BAAI/bge-m3` отдельно от общего sweep: dense retrieval через `sentence-transformers`, native dense+sparse retrieval через `FlagEmbedding`, и fusion с текущими сильными sparse-ветками.


In [ ]:
# 2c. Separate BGE-M3 retrieval experiment
# Default parameters are chosen for Colab GPU and this corpus:
# - line chunks 10/5 are the strongest current passage view and keep the index small (~2k chunks);
# - max_length=2048 limits very long legal lines without paying full 8192-token cost;
# - native BGE-M3 uses dense+sparse weights 0.7/0.3, then chunk->doc max aggregation;
# - strict CV remains the default, holdout should be used only after selecting 1-2 candidates.

import sys
from dataclasses import replace

from legal_hse.experiments import recall_candidate_experiments, run_suite, select_best_experiment

run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])

BGE_M3_MODE = globals().get("BGE_M3_MODE", "cv")  # "cv", "holdout", or "train"
BGE_M3_SEED = globals().get("BGE_M3_SEED", 42)
BGE_M3_N_SPLITS = globals().get("BGE_M3_N_SPLITS", 5)
BGE_M3_EVAL_KS = tuple(globals().get("BGE_M3_EVAL_KS", (5, 10, 20, 30, 50, 100)))
BGE_M3_EVAL_DEPTH = max(globals().get("BGE_M3_EVAL_DEPTH", 100), max(BGE_M3_EVAL_KS))
BGE_M3_BATCH_SIZE = int(globals().get("BGE_M3_BATCH_SIZE", 4))
BGE_M3_ST_BATCH_SIZE = int(globals().get("BGE_M3_ST_BATCH_SIZE", 8))
BGE_M3_MAX_LENGTH = int(globals().get("BGE_M3_MAX_LENGTH", 2048))
BGE_M3_RUN_CHAR = bool(globals().get("BGE_M3_RUN_CHAR", False))
BGE_M3_BASELINE = "rrf_sparse_deep_legal_lemma_char"

_extra_specs = recall_candidate_experiments(include_optional=False, include_bge_m3=True)
BGE_M3_EXTRA_EXPERIMENTS = []
for spec in _extra_specs:
    params = spec.params
    if spec.kind == "bge_m3_chunk":
        params = {
            **spec.params,
            "config": {
                **spec.params.get("config", {}),
                "batch_size": BGE_M3_BATCH_SIZE,
                "max_length": BGE_M3_MAX_LENGTH,
            },
        }
    elif spec.name == "dense_bge_m3_chunk_line_10_5_rd600":
        params = {
            **spec.params,
            "config": {
                **spec.params.get("config", {}),
                "batch_size": BGE_M3_ST_BATCH_SIZE,
                "max_seq_length": BGE_M3_MAX_LENGTH,
            },
        }
    BGE_M3_EXTRA_EXPERIMENTS.append(replace(spec, params=params))

BGE_M3_EXPERIMENT_NAMES = [
    BGE_M3_BASELINE,
    "dense_bge_m3_chunk_line_10_5_rd600",
    "bge_m3_dense_sparse_chunk_line_10_5_rd600",
    "rrf_sparse_bge_m3_line",
    "rrf_sparse_bge_m3_native_line",
    "quota_sparse_bge_m3_native_line_q8",
]
if BGE_M3_RUN_CHAR:
    BGE_M3_EXPERIMENT_NAMES.append("bge_m3_dense_sparse_chunk_char_1600_800_rd600")

print(f"Running {len(BGE_M3_EXPERIMENT_NAMES)} BGE-M3 experiments in mode={BGE_M3_MODE}")
for name in BGE_M3_EXPERIMENT_NAMES:
    print("-", name)

bge_m3_summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=BGE_M3_EXPERIMENT_NAMES,
    extra_experiments=BGE_M3_EXTRA_EXPERIMENTS,
    mode=BGE_M3_MODE,
    include_optional=False,
    seed=BGE_M3_SEED,
    n_splits=BGE_M3_N_SPLITS,
    eval_depth=BGE_M3_EVAL_DEPTH,
    eval_ks=BGE_M3_EVAL_KS,
    comparison_baseline=BGE_M3_BASELINE,
)

bge_cols = [
    "experiment",
    "status",
    "priority",
    "n_splits",
    "valid_recall@5_mean",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@50_mean",
    "valid_recall@100_mean",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "holdout_recall@5_mean",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@50_mean",
    "holdout_recall@100_mean",
]
bge_cols = [col for col in bge_cols if col in bge_m3_summary.columns]
display(bge_m3_summary[bge_cols].sort_values(bge_cols[4] if len(bge_cols) > 4 else "experiment", ascending=False))
print("Best BGE-M3 run:", select_best_experiment(bge_m3_summary))


## 2d. Rerank experiments

In [ ]:
# 2d. Cross-encoder rerank experiments
# This cell implements the rerank wave from deep-research-report.md:
# - build wide RRF/quota candidate lists;
# - select the best line-window chunks per candidate doc;
# - rerank query x chunk pairs with a multilingual cross-encoder;
# - compare rerank depth, number of chunks per doc, aggregation, and CE-only vs CE+candidate-score fusion;
# - write a submission for the best rerank config.

import hashlib
import json
import math
import re
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path.cwd()
else:
    PROJECT_DIR = Path(PROJECT_DIR)

if "run_cmd" not in globals():
    def run_cmd(cmd, *, cwd=None, check=True):
        print("$", " ".join(map(str, cmd)))
        result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if check and result.returncode != 0:
            raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
        return result

# Cross-encoder needs sentence-transformers; lemmatized BM25 chunk selector needs pymorphy3.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0", "pymorphy3>=2.0.6"])
RUN_OPTIONAL_DENSE = True

RERANK_ENABLE_E5_CANDIDATES = bool(globals().get("RERANK_ENABLE_E5_CANDIDATES", False))
RERANK_ENABLE_BGE_M3 = bool(globals().get("RERANK_ENABLE_BGE_M3", False))
if RERANK_ENABLE_BGE_M3:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "FlagEmbedding>=1.2.10"])

from legal_hse.chunking import ChunkConfig
from legal_hse.data import load_data
from legal_hse.experiments import (
    aggregate_validation_records,
    default_experiments,
    rank_queries,
    recall_candidate_experiments,
    select_best_experiment,
)
from legal_hse.metrics import dedupe_topk, evaluate_predictions
from legal_hse.retrievers.base import SearchResult
from legal_hse.retrievers.bm25 import BM25Config, BM25Retriever
from legal_hse.rerankers.cross_encoder import CrossEncoderConfig, CrossEncoderReranker
from legal_hse.splits import Split, make_group_holdout, make_group_kfold
from legal_hse.submission import write_submission


def _as_list(value):
    if isinstance(value, str):
        return [value]
    return list(value)


RERANK_MODE = globals().get("RERANK_MODE", "cv")  # "cv", "holdout", or "train"
RERANK_SEED = int(globals().get("RERANK_SEED", 42))
RERANK_N_SPLITS = int(globals().get("RERANK_N_SPLITS", 5))
RERANK_EVAL_KS = tuple(globals().get("RERANK_EVAL_KS", (5, 10, 20)))
RERANK_CANDIDATE_DEPTH = int(globals().get("RERANK_CANDIDATE_DEPTH", 100))
RERANK_DEPTHS = tuple(globals().get("RERANK_DEPTHS", (20, 50, 100)))
RERANK_CHUNKS_PER_DOC = tuple(globals().get("RERANK_CHUNKS_PER_DOC", (1, 2)))
RERANK_CHUNK_AGGS = tuple(globals().get("RERANK_CHUNK_AGGS", ("max", "top2_mean")))
RERANK_SCORE_MODES = tuple(globals().get("RERANK_SCORE_MODES", ("ce", "ce_plus_candidate")))
RERANK_MODEL_NAMES = _as_list(globals().get("RERANK_MODEL_NAMES", ["cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"]))
RERANK_BATCH_SIZE = int(globals().get("RERANK_BATCH_SIZE", 16))
RERANK_MAX_LENGTH = int(globals().get("RERANK_MAX_LENGTH", 512))
RERANK_DEVICE = globals().get("RERANK_DEVICE", None)  # e.g. "cuda" or "cpu"
RERANK_CHUNK_SEARCH_DEPTH = int(globals().get("RERANK_CHUNK_SEARCH_DEPTH", 2500))
RERANK_PAIR_CHAR_LIMIT = int(globals().get("RERANK_PAIR_CHAR_LIMIT", 3500))
RERANK_FALLBACK_DOC_CHARS = int(globals().get("RERANK_FALLBACK_DOC_CHARS", 2500))
RERANK_CREATE_SUBMISSION = bool(globals().get("RERANK_CREATE_SUBMISSION", True))

_default_candidates = [
    "rrf_sparse_deep_legal_lemma_char",
    "quota_sparse_legal_lemma_char_q10",
]
if RERANK_ENABLE_E5_CANDIDATES:
    _default_candidates += ["rrf_sparse_e5_line", "quota_sparse_e5_line_q8"]
if RERANK_ENABLE_BGE_M3:
    _default_candidates += ["rrf_sparse_bge_m3_native_line", "quota_sparse_bge_m3_native_line_q8"]
RERANK_CANDIDATE_EXPERIMENTS = _as_list(globals().get("RERANK_CANDIDATE_EXPERIMENTS", _default_candidates))
RERANK_CANDIDATE_DEPTH = max(RERANK_CANDIDATE_DEPTH, max(RERANK_DEPTHS), max(RERANK_EVAL_KS))


@dataclass(frozen=True)
class RerankConfig:
    name: str
    candidate_experiment: str
    model_name: str
    depth: int
    chunks_per_doc: int
    chunk_agg: str
    score_mode: str


def _slug(value: str, max_len: int = 80) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value[:max_len].strip("_") or "x"


def _merge_specs(base, extra):
    by_name = {}
    for spec in [*base, *extra]:
        by_name.setdefault(spec.name, spec)
    return list(by_name.values())


def _make_rerank_eval_folds(train: pd.DataFrame, *, mode: str, seed: int, n_splits: int):
    if mode == "holdout":
        split = make_group_holdout(train, seed=seed)
        holdout_df = train.iloc[split.valid_idx].reset_index(drop=True)
        return [(split.name, "holdout", holdout_df)]
    if mode == "cv":
        outer_split = make_group_holdout(train, seed=seed)
        cv_pool = train.iloc[outer_split.train_idx].reset_index(drop=True)
        folds = []
        for split in make_group_kfold(cv_pool, n_splits=n_splits):
            valid_df = cv_pool.iloc[split.valid_idx].reset_index(drop=True)
            folds.append((split.name, "valid", valid_df))
        return folds
    if mode == "train":
        return [("train_all", "train", train.reset_index(drop=True))]
    raise ValueError("RERANK_MODE must be one of: cv, holdout, train")


def _legal_bm25_config() -> dict:
    return {
        "k1": 1.5,
        "b": 0.75,
        "lemmatize": True,
        "preserve_legal_refs": True,
        "legal_stop_words": True,
        "min_len": 2,
    }


def _build_chunk_units(documents: pd.DataFrame, config: ChunkConfig) -> pd.DataFrame:
    from legal_hse.experiments import chunk_units

    return chunk_units(documents, config)


def _clean_pair_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:RERANK_PAIR_CHAR_LIMIT]


def _chunks_by_doc(chunk_ranking: list[SearchResult]) -> dict[str, list[SearchResult]]:
    grouped: dict[str, list[SearchResult]] = {}
    for item in chunk_ranking:
        grouped.setdefault(item.doc_id, []).append(item)
    return grouped


def _zscore(values: pd.Series) -> pd.Series:
    arr = values.astype(float)
    std = arr.std(ddof=0)
    if not math.isfinite(float(std)) or std < 1e-9:
        return pd.Series(np.zeros(len(arr)), index=arr.index)
    return (arr - arr.mean()) / std


def _aggregate_ce(scores: list[float], method: str) -> float:
    ordered = sorted([float(score) for score in scores], reverse=True)
    if not ordered:
        return float("-inf")
    if method == "max":
        return ordered[0]
    if method == "top2_mean":
        return float(np.mean(ordered[:2]))
    if method == "max_plus_second":
        return ordered[0] + (0.2 * ordered[1] if len(ordered) > 1 else 0.0)
    raise ValueError(f"Unknown chunk_agg: {method}")


def _prediction_lists_from_pairs(pair_frame: pd.DataFrame, qids: list[str], config: RerankConfig) -> list[list[str]]:
    subset = pair_frame[
        (pair_frame["candidate_rank"] <= config.depth)
        & (pair_frame["chunk_rank"] <= config.chunks_per_doc)
    ].copy()
    if subset.empty:
        return [[] for _ in qids]

    doc_rows = []
    for (qid, doc_id), group in subset.groupby(["qid", "doc_id"], sort=False):
        doc_rows.append(
            {
                "qid": qid,
                "doc_id": doc_id,
                "candidate_rank": int(group["candidate_rank"].min()),
                "candidate_score": float(group["candidate_score"].iloc[0]),
                "ce_score": _aggregate_ce(group["ce_score"].tolist(), config.chunk_agg),
            }
        )
    docs = pd.DataFrame(doc_rows)

    predictions: list[list[str]] = []
    for qid in qids:
        part = docs[docs["qid"].eq(qid)].copy()
        if part.empty:
            predictions.append([])
            continue
        if config.score_mode == "ce":
            part["final_score"] = part["ce_score"]
        elif config.score_mode == "ce_plus_candidate":
            part["final_score"] = 0.85 * _zscore(part["ce_score"]) + 0.15 * _zscore(part["candidate_score"])
        else:
            raise ValueError(f"Unknown score_mode: {config.score_mode}")
        part = part.sort_values(["final_score", "candidate_rank"], ascending=[False, True])
        predictions.append(part["doc_id"].astype(str).tolist())
    return predictions


def _query_hit_records(base_record: dict, eval_part: str, frame: pd.DataFrame, predictions: list[list[str]]) -> list[dict]:
    rows = []
    for qid, expected, predicted in zip(
        frame["qid"].astype(str).tolist(),
        frame["gold_doc_id"].astype(str).tolist(),
        predictions,
        strict=True,
    ):
        row = {
            "run_id": base_record["run_id"],
            "split": base_record["split"],
            "mode": base_record["mode"],
            "experiment": base_record["experiment"],
            "eval_part": eval_part,
            "qid": qid,
            "gold_doc_id": expected,
        }
        for k in RERANK_EVAL_KS:
            row[f"hit@{k}"] = int(expected in dedupe_topk(predicted, k))
        rows.append(row)
    return rows


def _metric_record(run_id: str, split_name: str, eval_part: str, experiment: str, description: str, frame: pd.DataFrame, predictions: list[list[str]], duration_sec: float, params: dict) -> tuple[dict, list[dict]]:
    gold = frame["gold_doc_id"].astype(str).tolist()
    record = {
        "run_id": run_id,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "split": split_name,
        "mode": RERANK_MODE,
        "experiment": experiment,
        "priority": "P0",
        "description": description,
        "params": json.dumps(params, ensure_ascii=False),
        "eval_part": eval_part,
        "n_eval": len(frame),
        "status": "ok",
        "duration_sec": float(duration_sec),
    }
    record.update(evaluate_predictions(gold, predictions, ks=RERANK_EVAL_KS))
    return record, _query_hit_records(record, eval_part, frame, predictions)


def _score_candidate_pairs(
    *,
    reranker: CrossEncoderReranker,
    queries: list[str],
    qids: list[str],
    candidate_rankings: list[list[SearchResult]],
    chunk_rankings: list[list[SearchResult]],
    max_depth: int,
    max_chunks: int,
    doc_text_by_id: dict[str, str],
) -> pd.DataFrame:
    meta_rows = []
    pairs = []
    for q_idx, (qid, query, candidates, chunks) in enumerate(zip(qids, queries, candidate_rankings, chunk_rankings, strict=True)):
        grouped_chunks = _chunks_by_doc(chunks)
        seen_docs: set[str] = set()
        for candidate_rank, candidate in enumerate(candidates[:max_depth], start=1):
            doc_id = str(candidate.doc_id)
            if doc_id in seen_docs:
                continue
            seen_docs.add(doc_id)
            selected_chunks = grouped_chunks.get(doc_id, [])[:max_chunks]
            if not selected_chunks:
                fallback = candidate.text or doc_text_by_id.get(doc_id, "")[:RERANK_FALLBACK_DOC_CHARS]
                selected_chunks = [
                    SearchResult(
                        doc_id=doc_id,
                        unit_id=f"{doc_id}::fallback",
                        score=float(candidate.score),
                        source="fallback_candidate_or_doc_intro",
                        text=fallback,
                    )
                ]
            for chunk_rank, chunk in enumerate(selected_chunks[:max_chunks], start=1):
                text = _clean_pair_text(chunk.text or "")
                meta_rows.append(
                    {
                        "qid": qid,
                        "q_idx": q_idx,
                        "doc_id": doc_id,
                        "candidate_rank": candidate_rank,
                        "candidate_score": float(candidate.score),
                        "chunk_rank": chunk_rank,
                        "chunk_score": float(chunk.score),
                        "chunk_id": chunk.unit_id,
                        "text_hash": hashlib.blake2b(text.encode("utf-8"), digest_size=8).hexdigest(),
                    }
                )
                pairs.append((query, text))

    if not pairs:
        return pd.DataFrame(meta_rows)

    if reranker.model is None:
        reranker.load()
    scores = reranker.model.predict(
        pairs,
        batch_size=RERANK_BATCH_SIZE,
        show_progress_bar=True,
    )
    pair_frame = pd.DataFrame(meta_rows)
    pair_frame["ce_score"] = np.asarray(scores, dtype=np.float32).reshape(-1)
    return pair_frame


def _get_reranker(model_name: str) -> CrossEncoderReranker:
    if model_name not in RERANKERS:
        RERANKERS[model_name] = CrossEncoderReranker(
            CrossEncoderConfig(
                model_name=model_name,
                batch_size=RERANK_BATCH_SIZE,
                max_length=RERANK_MAX_LENGTH,
                device=RERANK_DEVICE,
            )
        ).load()
    return RERANKERS[model_name]


def _make_rerank_configs(candidate_name: str, model_name: str) -> list[RerankConfig]:
    configs = []
    model_slug = _slug(model_name, max_len=42)
    candidate_slug = _slug(candidate_name, max_len=48)
    for depth in RERANK_DEPTHS:
        for chunks_per_doc in RERANK_CHUNKS_PER_DOC:
            for chunk_agg in RERANK_CHUNK_AGGS:
                if chunks_per_doc == 1 and chunk_agg != "max":
                    continue
                for score_mode in RERANK_SCORE_MODES:
                    name = f"rerank_{candidate_slug}_{model_slug}_d{depth}_c{chunks_per_doc}_{chunk_agg}_{score_mode}"
                    configs.append(
                        RerankConfig(
                            name=name,
                            candidate_experiment=candidate_name,
                            model_name=model_name,
                            depth=int(depth),
                            chunks_per_doc=int(chunks_per_doc),
                            chunk_agg=str(chunk_agg),
                            score_mode=str(score_mode),
                        )
                    )
    return configs


def _display_frame(frame: pd.DataFrame):
    try:
        display(frame)
    except NameError:
        print(frame.to_string(index=False))


paths = PROJECT_DIR
DATA = load_data(paths)
EXTRA_SPECS = recall_candidate_experiments(
    include_optional=RERANK_ENABLE_E5_CANDIDATES,
    include_bge_m3=RERANK_ENABLE_BGE_M3,
)
ALL_SPECS = _merge_specs(default_experiments(include_optional=RERANK_ENABLE_E5_CANDIDATES), EXTRA_SPECS)
SPECS_BY_NAME = {spec.name: spec for spec in ALL_SPECS}
missing = sorted(set(RERANK_CANDIDATE_EXPERIMENTS).difference(SPECS_BY_NAME))
if missing:
    raise ValueError(f"Unknown rerank candidate experiments: {missing}. Enable E5/BGE flags if needed.")

chunk_config = ChunkConfig(unit="line", size=10, overlap=5)
RERANK_CHUNK_UNITS = _build_chunk_units(DATA.documents, chunk_config)
RERANK_CHUNK_SELECTOR = BM25Retriever(
    "rerank_bm25_legal_lemma_chunk_line_10_5_raw",
    BM25Config(**_legal_bm25_config()),
).fit(RERANK_CHUNK_UNITS)
RERANK_CHUNK_SEARCH_DEPTH = min(RERANK_CHUNK_SEARCH_DEPTH, len(RERANK_CHUNK_UNITS))
DOC_TEXT_BY_ID = DATA.documents.set_index("doc_id")["text"].fillna("").astype(str).to_dict()
RERANKERS: dict[str, CrossEncoderReranker] = {}
RERANK_CONFIGS_BY_NAME: dict[str, RerankConfig] = {}

RUN_ID = datetime.now(timezone.utc).strftime("rerank_%Y%m%dT%H%M%SZ")
records: list[dict] = []
query_hit_rows: list[dict] = []
folds = _make_rerank_eval_folds(DATA.train, mode=RERANK_MODE, seed=RERANK_SEED, n_splits=RERANK_N_SPLITS)

print("Rerank suite")
print("mode:", RERANK_MODE)
print("candidate experiments:", RERANK_CANDIDATE_EXPERIMENTS)
print("models:", RERANK_MODEL_NAMES)
print("depths:", RERANK_DEPTHS, "chunks_per_doc:", RERANK_CHUNKS_PER_DOC, "score_modes:", RERANK_SCORE_MODES)
print("chunk selector units:", len(RERANK_CHUNK_UNITS), "search_depth:", RERANK_CHUNK_SEARCH_DEPTH)

for split_name, eval_part, eval_frame in folds:
    queries = eval_frame["question"].astype(str).tolist()
    qids = eval_frame["qid"].astype(str).tolist()
    print(f"\nFold {split_name}: {len(queries)} queries")

    chunk_rankings = RERANK_CHUNK_SELECTOR.search(queries, top_k=RERANK_CHUNK_SEARCH_DEPTH)
    rank_cache: dict[str, list[list[SearchResult]]] = {}

    for candidate_name in RERANK_CANDIDATE_EXPERIMENTS:
        candidate_started = time.time()
        candidate_rankings = rank_queries(
            SPECS_BY_NAME[candidate_name],
            ALL_SPECS,
            DATA.documents,
            queries,
            top_k=RERANK_CANDIDATE_DEPTH,
            cache=rank_cache,
        )
        baseline_predictions = [[item.doc_id for item in ranking[:RERANK_CANDIDATE_DEPTH]] for ranking in candidate_rankings]
        baseline_name = f"candidate_{_slug(candidate_name, max_len=70)}"
        base_record, base_hits = _metric_record(
            RUN_ID,
            split_name,
            eval_part,
            baseline_name,
            f"No-rerank candidate baseline from {candidate_name}.",
            eval_frame,
            baseline_predictions,
            time.time() - candidate_started,
            {"candidate_experiment": candidate_name, "depth": RERANK_CANDIDATE_DEPTH},
        )
        records.append(base_record)
        query_hit_rows.extend(base_hits)

        for model_name in RERANK_MODEL_NAMES:
            print(f"Scoring {candidate_name} with {model_name}")
            pair_started = time.time()
            pair_frame = _score_candidate_pairs(
                reranker=_get_reranker(model_name),
                queries=queries,
                qids=qids,
                candidate_rankings=candidate_rankings,
                chunk_rankings=chunk_rankings,
                max_depth=max(RERANK_DEPTHS),
                max_chunks=max(RERANK_CHUNKS_PER_DOC),
                doc_text_by_id=DOC_TEXT_BY_ID,
            )
            scoring_sec = time.time() - pair_started

            for config in _make_rerank_configs(candidate_name, model_name):
                RERANK_CONFIGS_BY_NAME[config.name] = config
                predictions = _prediction_lists_from_pairs(pair_frame, qids, config)
                record, hits = _metric_record(
                    RUN_ID,
                    split_name,
                    eval_part,
                    config.name,
                    (
                        f"Cross-encoder rerank of {candidate_name}: depth={config.depth}, "
                        f"chunks={config.chunks_per_doc}, agg={config.chunk_agg}, score={config.score_mode}."
                    ),
                    eval_frame,
                    predictions,
                    scoring_sec,
                    asdict(config),
                )
                records.append(record)
                query_hit_rows.extend(hits)

raw = pd.DataFrame(records)
query_hits = pd.DataFrame(query_hit_rows)
comparison_baseline = f"candidate_{_slug(RERANK_CANDIDATE_EXPERIMENTS[0], max_len=70)}"
RERANK_SUMMARY = aggregate_validation_records(
    raw,
    query_hits=query_hits,
    comparison_baseline=comparison_baseline,
    metric_columns=[f"recall@{k}" for k in RERANK_EVAL_KS],
)

reports_dir = PROJECT_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
raw_path = reports_dir / f"folds_{RUN_ID}.csv"
query_hits_path = reports_dir / f"query_hits_{RUN_ID}.csv"
summary_path = reports_dir / f"summary_{RUN_ID}.csv"
latest_path = reports_dir / "rerank_summary_latest.csv"
raw.to_csv(raw_path, index=False)
query_hits.to_csv(query_hits_path, index=False)
RERANK_SUMMARY.to_csv(summary_path, index=False)
RERANK_SUMMARY.to_csv(latest_path, index=False)

sort_col = next(
    col
    for col in ["holdout_recall@5_mean", "valid_recall@5_mean", "train_recall@5_mean"]
    if col in RERANK_SUMMARY.columns
)
visible_cols = [
    "experiment",
    "status",
    "n_splits",
    "valid_recall@5_mean",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "holdout_recall@5_mean",
    "holdout_recall@5_delta_vs_baseline",
    "holdout_recall@5_wins_vs_baseline",
    "holdout_recall@5_losses_vs_baseline",
    "duration_sec",
]
visible_cols = [col for col in visible_cols if col in RERANK_SUMMARY.columns]
ranked_rerank_summary = RERANK_SUMMARY[visible_cols].sort_values(["status", sort_col], ascending=[True, False])
_display_frame(ranked_rerank_summary.head(50))

rerank_only = RERANK_SUMMARY[RERANK_SUMMARY["experiment"].astype(str).str.startswith("rerank_")].copy()
BEST_RERANK_EXPERIMENT = select_best_experiment(rerank_only, metric=sort_col)
BEST_RERANK_OR_CANDIDATE = select_best_experiment(RERANK_SUMMARY, metric=sort_col)
print("BEST_RERANK_EXPERIMENT =", BEST_RERANK_EXPERIMENT)
print("BEST_RERANK_OR_CANDIDATE =", BEST_RERANK_OR_CANDIDATE)
print("Saved rerank summary:", summary_path)


def _make_rerank_submission(config: RerankConfig) -> Path:
    test_queries = DATA.test["question"].astype(str).tolist()
    test_qids = DATA.test["qid"].astype(str).tolist()
    test_chunk_rankings = RERANK_CHUNK_SELECTOR.search(test_queries, top_k=RERANK_CHUNK_SEARCH_DEPTH)
    test_rank_cache: dict[str, list[list[SearchResult]]] = {}
    test_candidate_rankings = rank_queries(
        SPECS_BY_NAME[config.candidate_experiment],
        ALL_SPECS,
        DATA.documents,
        test_queries,
        top_k=max(RERANK_CANDIDATE_DEPTH, config.depth),
        cache=test_rank_cache,
    )
    test_pair_frame = _score_candidate_pairs(
        reranker=_get_reranker(config.model_name),
        queries=test_queries,
        qids=test_qids,
        candidate_rankings=test_candidate_rankings,
        chunk_rankings=test_chunk_rankings,
        max_depth=config.depth,
        max_chunks=config.chunks_per_doc,
        doc_text_by_id=DOC_TEXT_BY_ID,
    )
    test_predictions = _prediction_lists_from_pairs(test_pair_frame, test_qids, config)
    output_path = PROJECT_DIR / "submissions" / f"submission_{config.name}.csv"
    return write_submission(
        test_qids,
        test_predictions,
        output_path,
        test=DATA.test,
        documents=DATA.documents,
        top_k=5,
    )


if RERANK_CREATE_SUBMISSION:
    RERANK_SUBMISSION_PATH = _make_rerank_submission(RERANK_CONFIGS_BY_NAME[BEST_RERANK_EXPERIMENT])
    print("Rerank submission:", RERANK_SUBMISSION_PATH)
    _display_frame(pd.read_csv(RERANK_SUBMISSION_PATH).head(15))


In [ ]:
# 2. Все эксперименты
from legal_hse.experiments import run_suite, select_best_experiment

MODE = globals().get("MODE", "cv")  # "holdout", "cv" or "train"
EXPERIMENT_NAMES = globals().get("EXPERIMENT_NAMES", None)  # None runs default suite
SEED = globals().get("SEED", 42)
N_SPLITS = globals().get("N_SPLITS", 5)

summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=EXPERIMENT_NAMES,
    mode=MODE,
    include_optional=RUN_OPTIONAL_DENSE,
    seed=SEED,
    n_splits=N_SPLITS,
    eval_depth=50,
)

cols = [
    "experiment",
    "status",
    "n_splits",
    "train_recall@5_mean",
    "train_recall@5_std",
    "valid_recall@5_mean",
    "valid_recall@5_micro",
    "valid_recall@5_se",
    "valid_recall@5_std",
    "valid_recall@5_delta_vs_baseline",
    "valid_recall@5_wins_vs_baseline",
    "valid_recall@5_losses_vs_baseline",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@50_mean",
    "holdout_recall@5_mean",
    "holdout_recall@5_micro",
    "holdout_recall@5_se",
    "holdout_recall@5_std",
    "holdout_recall@5_delta_vs_baseline",
    "holdout_recall@5_wins_vs_baseline",
    "holdout_recall@5_losses_vs_baseline",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@50_mean",
]
visible_cols = [c for c in cols if c in summary.columns]
sort_col = next(c for c in ["holdout_recall@5_mean", "valid_recall@5_mean", "train_recall@5_mean"] if c in summary.columns)
display(summary[visible_cols].sort_values(["status", sort_col], ascending=[True, False]))
BEST_EXPERIMENT = select_best_experiment(summary)
print("BEST_EXPERIMENT =", BEST_EXPERIMENT)

## 3. Загрузка результатов на GitHub

Введите `GITHUB_USERNAME`, `GIT_EMAIL`, `SSH_PRIVATE_KEY_B64`. Ключ должен иметь write-доступ к репозиторию.

In [ ]:
# 3. Загрузка результатов [метрик] на GitHub
import base64
import getpass
import os
import subprocess
from pathlib import Path

GITHUB_USERNAME = input("GITHUB_USERNAME: ").strip()
GIT_EMAIL = input("GIT_EMAIL: ").strip()
SSH_PRIVATE_KEY_B64 = getpass.getpass("SSH_PRIVATE_KEY_B64: ").strip()
SSH_REMOTE_URL = ""  # optional: git@github.com:<user>/<repo>.git

ssh_dir = Path.home() / ".ssh"
ssh_dir.mkdir(mode=0o700, exist_ok=True)
key_path = ssh_dir / "id_ed25519"
key_path.write_bytes(base64.b64decode(SSH_PRIVATE_KEY_B64))
key_path.chmod(0o600)
subprocess.run(["ssh-keyscan", "github.com"], stdout=(ssh_dir / "known_hosts").open("a"), check=True)

subprocess.run(["git", "config", "user.name", GITHUB_USERNAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

if not SSH_REMOTE_URL and REPO_URL.startswith("https://github.com/"):
    repo_path = REPO_URL.split("https://github.com/", 1)[1]
    repo_path = repo_path[:-4] if repo_path.endswith(".git") else repo_path
    SSH_REMOTE_URL = f"git@github.com:{repo_path}.git"
if SSH_REMOTE_URL:
    subprocess.run(["git", "remote", "set-url", "origin", SSH_REMOTE_URL], check=True)

paths_to_add = ["reports/metrics", "reports/summary_latest.csv"]
paths_to_add += [str(path) for path in Path("reports").glob("summary_*.csv")]
subprocess.run(["git", "add", *paths_to_add], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
if status:
    subprocess.run(["git", "commit", "-m", "Add retrieval experiment metrics"], check=True)
    subprocess.run(["git", "push", "origin", f"HEAD:{BRANCH}"], check=True)
    print("Metrics pushed to GitHub.")
else:
    print("No metric changes to push.")

## 4. Формирование submission файла

По умолчанию используется лучший experiment name из блока 2. Его можно заменить вручную.

In [ ]:
# 4. Формирование submission файла
import pandas as pd
from legal_hse.experiments import create_submission

FINAL_EXPERIMENT = globals().get("BEST_EXPERIMENT", "rrf_bm25_doc_chunk")
SUBMISSION_PATH = create_submission(
    data_dir=PROJECT_DIR,
    experiment_name=FINAL_EXPERIMENT,
    output_path=PROJECT_DIR / "submissions" / f"submission_{FINAL_EXPERIMENT}.csv",
    include_optional=RUN_OPTIONAL_DENSE,
    top_k=5,
)
print("Submission:", SUBMISSION_PATH)
display(pd.read_csv(SUBMISSION_PATH).head(15))